# ĐỒ ÁN IS403 — Ô NHIỄM KHÔNG KHÍ TẠI TP.HCM
### Nhóm 4 — Môn Phân tích dữ liệu kinh doanh (IS403)


## 1. Bối cảnh & vấn đề thực tế

TP.HCM là đô thị lớn ở Đông Nam Á, ô nhiễm bụi mịn (PM2.5) thường xuyên **vượt nhiều lần** khuyến nghị của WHO và Quy chuẩn Việt Nam (QCVN 05:2023) — đặc biệt vào mùa khô. Bụi mịn ảnh hưởng trực tiếp tới sức khoẻ hô hấp, và gián tiếp tới nhiều hoạt động ngoài trời.

**Vấn đề:** hiện tại các đơn vị chịu ảnh hưởng (tổ chức sự kiện ngoài trời, công ty du lịch, cơ sở y tế, doanh nghiệp bán sản phẩm bảo vệ hô hấp) **không có công cụ nào giúp họ biết trước** — dựa trên dữ liệu thật — thời điểm nào sắp tới sẽ ô nhiễm nặng, để chủ động điều chỉnh kế hoạch.

→ Đề tài này xây dựng phân tích + mô hình dựa trên dữ liệu thật để lấp khoảng trống đó.

## 2. Câu hỏi kinh doanh (Business Question)

**★ Câu hỏi chính:**
> TP.HCM sẽ có những thời điểm nào ô nhiễm không khí NẶNG sắp tới, nhân tố nào tác động mạnh nhất — để các đơn vị chịu ảnh hưởng (sự kiện ngoài trời, du lịch, y tế, bán sản phẩm bảo vệ hô hấp) chủ động CẢNH BÁO và ĐIỀU CHỈNH hoạt động?

**5 câu hỏi con — mỗi câu là 1 phần phân tích:**

| # | Câu hỏi con | Phương pháp | Ai làm | File |
|---|---|---|---|---|
| ① | AQI phân bố thế nào theo **tháng/mùa**?  | Mô tả + visualize | Thành viên 1  | `02_thong_ke_mo_ta.ipynb`  |
| ② | Mùa khô vs mưa, ngày thường vs cuối tuần có khác? | t-test / ANOVA + Tukey HSD | Thành viên 1  | `03_kiem_dinh_hoi_quy.ipynb`  |
| ③ | Nhân tố thời tiết nào ảnh hưởng AQI mạnh nhất? | Hồi quy + VIF | Thành viên 1  | `03_kiem_dinh_hoi_quy.ipynb`  |
| ④ | Dự báo AQI mấy ngày tới? | SARIMA / ARIMAX | Thành viên 3 | (ngoài phạm vi 3 file này) |
| ⑤ | Cảnh báo sớm "ngày ô nhiễm nặng"? | Logistic / XGBoost / SVM | Thành viên 2 | (ngoài phạm vi 3 file này) |


## 3. Dữ liệu sử dụng (tóm tắt — Bước 1 bên dưới sẽ giải thích chi tiết cách lấy)

| | |
|---|---|
| **Biến Y (mục tiêu)** | `pm25` — nồng độ bụi mịn PM2.5 (µg/m³), đại diện cho AQI |
| **Biến X** | nhiệt độ, độ ẩm, gió (tốc độ + **hướng**), lượng mưa, **số ngày kể từ cơn mưa gần nhất**, lớp biên khí quyển, áp suất, tháng, thứ trong tuần, mùa, **ngày lễ / Tết**, các khí **CO, NO, NO₂, SO₂, O₃, PM10** (ước tính từ mô hình CAMS, không phải cảm biến) |
| **Thang đo** | ngoài thang Mỹ (US EPA) còn tính theo **chuẩn do Việt Nam công bố**: VN_AQI (QĐ 1459/QĐ-TCMT), QCVN 05:2023/BTNMT, kèm mốc WHO 2021 |
| **Nguồn** | OpenAQ (trạm CMT8, TP.HCM) + Open-Meteo (thời tiết) — 2 API thật, có thể tự kiểm chứng |
| **Đơn vị thời gian** | theo **NGÀY** (không phải giờ) |
| **Phạm vi** | 19/11/2024 → hiện tại, ~629 ngày |

**Giới hạn đã biết trước (sẽ chứng minh chi tiết ở Bước 1 & 2):**
- Chỉ dùng được **PM2.5**, không có PM10 (cảm biến pm10 của trạm đã hỏng từ 14/01/2025)
- Dữ liệu từ **1 điểm đo** (trạm CMT8), không đại diện toàn TP.HCM
- Mức **theo ngày**, không phân tích được biến động trong 1 ngày (theo giờ)

### Nguồn dữ liệu và văn bản dùng trong notebook này (lấy gì, dùng làm gì)

| Nguồn | Link | Lấy gì | Dùng làm gì |
|---|---|---|---|
| **OpenAQ API v3** — trạm CMT8 (location 3276359, nhà cung cấp AirGradient) | [docs.openaq.org](https://docs.openaq.org) · [trạm CMT8](https://explore.openaq.org/locations/3276359) | PM2.5, nhiệt độ, độ ẩm **trung bình theo ngày** và % số giờ có số đo | Biến Y `pm25`; biến X `temperature`, `humidity`; các cột `_cov` để kiểm tra chất lượng |
| **Open-Meteo Historical Weather API** (dữ liệu tái phân tích ERA5/ECMWF) | [tài liệu](https://open-meteo.com/en/docs/historical-weather-api) | Gió tối đa, lượng mưa, áp suất, hướng gió (theo ngày); lớp biên khí quyển (theo giờ) | Biến X thời tiết (`wind`, `rain`, `ap_suat`, `pbl_sang_m`), hướng gió, và `ngay_tu_mua` |
| **Open-Meteo Air Quality API** (mô hình CAMS, Copernicus) | [tài liệu](https://open-meteo.com/en/docs/air-quality-api) · [CAMS](https://atmosphere.copernicus.eu) | CO, NO, NO₂, SO₂, O₃, PM10 theo giờ; lưới toàn cầu khoảng 45 km (0,4°) | Các cột `_cams` (chỉ báo nguồn phát thải); đối chiếu PM2.5 với trạm ở notebook 01 |
| **Quyết định 1459/QĐ-TCMT** (12/11/2019, Tổng cục Môi trường) | [văn bản gốc](https://cem.gov.vn/storage/news_file_attach/QD%201459%20TCMT%20ngay%2012.11.2019%20AQI.pdf) | Bảng 2 (mốc PM2.5 24 giờ), Công thức 2 (nội suy), tên các mức, Bảng 4–5 (ảnh hưởng sức khỏe, khuyến nghị) | `vn_aqi_pm25`, `vn_aqi_cat`, `kem_tro_len` |
| **QCVN 05:2023/BTNMT** (Thông tư 01/2023/TT-BTNMT) | [văn bản gốc](https://thuvienmoitruong.vn/wp-content/uploads/2025/05/QCVN05_2023_BTNMT_919891.pdf) | Bảng 1: PM2.5 24 giờ 50 µg/m³ (45 từ 01/01/2026), trung bình năm 25 | `qcvn_24h`, `vuot_qcvn` |
| **WHO Global Air Quality Guidelines 2021** | [trang WHO](https://www.who.int/publications/i/item/9789240034228) | PM2.5: 15 µg/m³ (24 giờ), 5 µg/m³ (năm) | `vuot_who` (đối chiếu chuẩn quốc tế) |
| **Bộ luật Lao động 45/2019/QH14, Điều 112** và các thông báo nghỉ lễ từng năm | xem bảng ở Bước G | Số ngày nghỉ lễ và lịch nghỉ liền mạch từng kỳ | `ky_nghi_le`, `tet` |
| **Tài liệu khí hậu Nam Bộ** | [PLO](https://plo.vn/sau-ngay-5-5-tphcm-va-nam-bo-buoc-vao-mua-mua-post480943.html) · [CafeF](https://cafef.vn/mua-mua.html) | Mùa mưa Nam Bộ bắt đầu khoảng 5–15/5 và kéo đến khoảng trung tuần tháng 11 | Cách chia mùa `mua` (xem ghi chú dưới) |

**Ghi chú về cột `mua`:** nhóm quy ước **tháng 5–10 là mùa mưa, các tháng còn lại là mùa khô** (chia theo nguyên tháng cho đơn giản). Tài liệu khí hậu nêu mùa mưa kéo đến khoảng trung tuần tháng 11, nên tháng 11 là tháng chuyển mùa. Nhóm giữ tháng 11 ở mùa khô vì PM2.5 tháng 11 (khoảng 36,8 µg/m³) giống tháng 10 (35,3) và các tháng mùa khô hơn là các tháng mùa mưa. Đây là **quy ước của nhóm**, không phải số liệu khí tượng chính thức, và các nguồn trên là báo điện tử nên nên nói rõ khi trình bày.

### Tài liệu khoa học giải thích vì sao chọn các biến (cơ chế) — có trích dẫn


| Biến / nhóm | Nhận định | Tài liệu nói gì | Dùng làm gì |
|---|---|---|---|
| `pm25`, `mua` | PM2.5 ở TP.HCM thấp nhất mùa mưa, cao nhất mùa khô | T1: đo PM2.5 hằng ngày tại 5 điểm TP.HCM (03/2017–03/2018), tháng thấp nhất rơi vào mùa mưa (7–8) và cao nhất vào mùa khô (11–2) (đối chiếu toàn văn, tr.1 và tr.6); T2 (tr.1, tr.12) cũng nêu mùa mưa thấp hơn do rửa trôi. Hai bài chia mùa mưa là tháng 5–11, còn cột `mua` của nhóm quy ước tháng 5–10 | Cơ sở để dùng `mua`, `month` làm biến mùa vụ |
| `wind` | Gió mạnh làm PM2.5 giảm | T1: PM2.5 tương quan nghịch với tốc độ gió | Giải thích dấu âm của hệ số `wind` |
| `rain`, `ngay_tu_mua` | Mưa rửa trôi PM2.5, nhưng chỉ khi mưa đủ lớn | T5: PM2.5 giảm khoảng 21% trong 1 giờ sau khi mưa bắt đầu; tỷ lệ rửa trôi khoảng 2% với mưa nhẹ, khoảng 28% với mưa vừa, khoảng 27% với mưa to | Cơ sở chọn ngưỡng **1 mm** để coi là "có mưa" ở `ngay_tu_mua` (mưa nhẹ hầu như không rửa bụi) |
| `pbl_sang_m` | Lớp biên khí quyển càng thấp, PM2.5 càng cao | T3: PM2.5 có quan hệ nghịch với độ cao lớp biên (tuyến tính trong các đợt mù khô, dạng mũ âm trong các đợt sương mù lẫn bụi); T4: một năm đo ở Almaty cũng thấy quan hệ nghịch | Giải thích dấu âm của `pbl_sang_m` |
| `no_cams`, `no2_cams` | NO, NO₂ là chỉ báo khí thải giao thông | T6: NO₂ chủ yếu sinh ra từ đốt nhiên liệu (ô tô, xe tải, xe buýt, nhà máy điện); T13 (báo): xe máy chiếm khoảng 69% NOx phát thải ở TP.HCM | Diễn giải các cột này là chỉ báo giao thông |
| `co_cams` | CO là chỉ báo đốt cháy nhiên liệu | T7: nguồn lớn nhất của CO ngoài trời là xe cộ và máy móc đốt nhiên liệu hóa thạch; T13 (báo): xe máy chiếm khoảng 98% CO phát thải ở TP.HCM | Diễn giải là chỉ báo giao thông và đốt cháy |
| `so2_cams` | SO₂ là chỉ báo đốt nhiên liệu hóa thạch (công nghiệp, nhà máy điện, và cả giao thông dùng nhiên liệu chứa lưu huỳnh) | T8: nguồn lớn nhất của SO₂ là đốt nhiên liệu hóa thạch ở nhà máy điện và cơ sở công nghiệp, nguồn nhỏ hơn gồm phương tiện dùng nhiên liệu nhiều lưu huỳnh; SO₂ có thể phản ứng tạo hạt nhỏ góp vào bụi. T13 (báo): xe máy chiếm khoảng 38% SO₂ phát thải ở TP.HCM | Diễn giải là chỉ báo **đốt nhiên liệu hóa thạch** (không chỉ riêng công nghiệp); giải thích vì sao có thể đi cùng PM2.5 |
| `o3_cams` | O₃ không thải trực tiếp, hình thành từ NOx và VOC dưới nắng | T9: O₃ tạo ra bởi phản ứng giữa NOx và VOC khi có ánh nắng, dễ cao nhất vào ngày nóng nắng | Diễn giải là chỉ báo nắng và hóa học khí quyển; lấy thêm `o3_max_cams` vì O₃ chỉ đạt đỉnh buổi trưa |
| `ky_nghi_le`, `tet` | Dịp lễ, Tết làm giảm hoạt động giao thông | T11 (Thâm Quyến): trong Tết Nguyên đán Trung Quốc, NOx, VOC và carbon đen giảm 50–80% bất kể thời tiết, còn PM2.5 tổng thay đổi ít hơn nhiều; T10 (Bắc Kinh): các chất sơ cấp giảm (NO giảm 54%) nhưng PM2.5 tổng không giảm; T12 (báo): trước Tết 2022 không khí TP.HCM xấu đi do giao thông tăng và sương mù | Căn cứ đưa vào biến lễ/Tết, và cảnh báo rằng tác động lên PM2.5 tổng có thể nhỏ |
| Mức PM2.5 của dữ liệu | Trung bình 32,4 µg/m³ cùng bậc với các nghiên cứu trước | T1: trung bình năm 36,3 ± 13,7 (2017–2018); T2: trung bình 28,44 ± 11,55 (09/2019–08/2020) | Đối chiếu độ lớn số liệu (notebook 01) |

**Danh sách tài liệu (mã T dùng thống nhất trong các notebook và file Word):**

| Mã | Tài liệu | Mức xác minh |
|---|---|---|
| T1 | To Thi Hien và cộng sự (2019), *Current Status of Fine Particulate Matter (PM2.5) in Vietnam's Most Populous City, Ho Chi Minh City*, Aerosol and Air Quality Research 19: 2239–2251, DOI 10.4209/aaqr.2018.12.0471. [Link](https://aaqr.org/articles/aaqr-18-12-oa-0471) | Đã tải toàn văn (PDF) và đối chiếu: tóm tắt tr.1; mục Monthly and Seasonal Variations tr.6 |
| T2 | Tran và cộng sự (2023), *Annual Variation of PM2.5 Chemical Composition in Ho Chi Minh City, Vietnam Including the COVID-19 Outbreak Period*, Aerosol and Air Quality Research 23, DOI 10.4209/aaqr.220312. [Link](https://link.springer.com/10.4209/aaqr.220312) | Đã tải toàn văn và đối chiếu: PM2.5 mùa khô cao hơn mùa mưa (tr.1, tr.12). Lưu ý bài chia mùa khô tháng 12–4, mùa mưa tháng 5–11 |
| T3 | Luan, Guo, Guo, Zhang (2018), *Quantifying the relationship between PM2.5 concentration, visibility and planetary boundary layer height for long-lasting haze and fog–haze mixed events in Beijing*, Atmospheric Chemistry and Physics 18: 203–225, DOI 10.5194/acp-18-203-2018. [Link](https://acp.copernicus.org/articles/18/203/2018/) | Đã tải toàn văn và đối chiếu (tr.1, tr.6) |
| T4 | Tursumbayeva và cộng sự (2022), *Planetary Boundary Layer and its Relationship with PM2.5 Concentrations in Almaty, Kazakhstan*. [Link](https://nur.nu.edu.kz/handle/123456789/10488) | Đã tải toàn văn và đối chiếu: quan hệ nghịch PM2.5 với lớp biên (tr.1, tr.10). AAQR 22(8): 210294, DOI 10.4209/aaqr.210294 |
| T5 | Fujino và Miyamoto (2022), *PM2.5 decrease with precipitation as revealed by single-point ground-based observation*, Atmospheric Science Letters 23(7): e1088. [Link](https://keio.elsevierpure.com/en/publications/pmsub25sub-decrease-with-precipitation-as-revealed-by-single-poin/) | Đã mở và đọc phần tóm tắt trên trang kho lưu trữ của Đại học Keio (đủ các số liệu 20,99% / 2,03% / 28,15% / 26,75% dùng trong đồ án); toàn văn không tải được nên chưa đối chiếu phần phương pháp chi tiết |
| T6 | US EPA, *Basic Information about NO2*. [Link](https://www.epa.gov/no2-pollution/basic-information-about-no2) | Đã tải trang và đối chiếu từng câu |
| T7 | US EPA, *Basic Information about Carbon Monoxide (CO) Outdoor Air Pollution*. [Link](https://www.epa.gov/co-pollution/basic-information-about-carbon-monoxide-co-outdoor-air-pollution) | Đã tải trang và đối chiếu từng câu |
| T8 | US EPA, *Sulfur Dioxide Basics*. [Link](https://www.epa.gov/so2-pollution/sulfur-dioxide-basics) | Đã tải trang và đối chiếu từng câu |
| T9 | US EPA, *Ground-level Ozone Basics*. [Link](https://www.epa.gov/ground-level-ozone-pollution/ground-level-ozone-basics) | Đã tải trang và đối chiếu từng câu |
| T10 | Jiang, Sun, Wang, Yin (2015), *Aerosol composition and sources during the Chinese Spring Festival: fireworks, secondary aerosol, and holiday effects*, Atmospheric Chemistry and Physics 15: 6023–6034, DOI 10.5194/acp-15-6023-2015. [Link](https://acp.copernicus.org/articles/15/6023/2015/acp-15-6023-2015.html) | Đã tải toàn văn và đối chiếu (tr.1, tr.8, tr.9) |
| T11 | Wang, Huang, Zhu, Cao, Zhang, He (2017), *Differentiating local and regional sources of Chinese urban air pollution based on the effect of the Spring Festival*, Atmospheric Chemistry and Physics 17: 9103–9114, DOI 10.5194/acp-17-9103-2017. [Link](https://acp.copernicus.org/articles/17/9103/2017/) | Đã tải toàn văn và đối chiếu (tr.1, tr.3) |
| T12 | Tuoi Tre News (07/01/2022), *Ho Chi Minh City's sky goes dull gray due to heavy air pollution*. [Link](https://tuoitrenews.vn/news/society/20220107/ho-chi-minh-citys-sky-goes-dull-gray-due-to-heavy-air-pollution/65117.html) | Đã đọc toàn bài; là bài báo (không phải nghiên cứu), nêu lời chuyên gia |
| T13 | Báo Nông nghiệp và Môi trường (26/04/2025), *Addressing air pollution: Vietnam tightens emission standards for transportation*, nêu tỷ trọng xe máy trong phát thải ở TP.HCM tại hội thảo khoa học quốc gia 24/04/2025 (số liệu đặt ngay sau phần phân tích của PGS.TS. Hồ Quốc Bằng, ĐHQG-HCM; bài không ghi rõ người nêu). [Link](https://van.nongnghiepmoitruong.vn/addressing-air-pollution-vietnam-tightens-emission-standards-for-transportation-d750171.html) | Đã đọc toàn bài; là bài báo (thứ cấp), số liệu nằm ngay sau phần phân tích của PGS.TS. Hồ Quốc Bằng |

**Lưu ý trung thực:** các tài liệu về Tết/lễ và về lớp biên là nghiên cứu ở thành phố khác (Thâm Quyến, Bắc Kinh, Almaty) vì chưa tìm được nghiên cứu định lượng tương ứng cho TP.HCM; chỉ dùng để giải thích **cơ chế**, không dùng để khẳng định độ lớn tác động ở TP.HCM.

# BƯỚC 1: TỰ THU THẬP DỮ LIỆU 


**Mục tiêu:** tạo ra file `aqi_hcm_daily.csv` — dữ liệu PM2.5 + nhiệt độ + độ ẩm (từ trạm CMT8, TP.HCM) ghép với gió + mưa (từ Open-Meteo), tổng hợp theo từng ngày.


## Ba thư viện cần dùng

`requests` gọi API (gửi yêu cầu qua Internet, nhận dữ liệu về); `pandas` xử lý bảng (như Excel bằng code); `numpy` tính toán và tạo điều kiện lọc.

In [2]:
import os          # thao tac voi bien moi truong, duong dan file
import getpass     # nhap key ma khong hien ra man hinh
import requests     # goi API qua Internet (gui yeu cau, nhan phan hoi)
import pandas as pd  # xu ly bang du lieu (giong Excel bang code)
import numpy as np    # tinh toan so hoc, tao dieu kien loc

print("Da nap xong 3 thu vien.")

Da nap xong 3 thu vien.


## Gắn API key

Ô dưới dùng `getpass`: key không hiện ra màn hình và không bị lưu vào file notebook.

In [2]:
# getpass an key khoi man hinh -> key khong bi luu vao file notebook
API_KEY = getpass.getpass("Dan OpenAQ API key roi nhan Enter (go se khong hien ra man hinh): ")

# HEADERS se duoc dinh kem vao MOI yeu cau goi OpenAQ ben duoi,
# giong nhu xuat trinh the thu vien moi lan muon sach
HEADERS = {"X-API-Key": API_KEY}

print("Da nhan key, do dai:", len(API_KEY), "ky tu")

Dan OpenAQ API key roi nhan Enter (go se khong hien ra man hinh):  ········


Da nhan key, do dai: 64 ky tu


## Cấu hình: trạm nào, ở đâu

- `LOCATION_ID = 3276359` → mã số của trạm CMT8 trên hệ thống OpenAQ (mỗi trạm quan trắc có 1 mã số duy nhất, giống mã số sinh viên).
- `LAT(Latitude - Vĩ độ), LON(Longitude - Kinh độ)` → toạ độ của trạm này. Ta sẽ dùng lại đúng toạ độ này ở bước kéo thời tiết, để đảm bảo lấy gió/mưa **đúng tại vị trí trạm**, không lấy lung tung chỗ khác.

In [3]:
LOCATION_ID = 3276359          # ma so tram CMT8 tren he thong OpenAQ
LAT, LON = 10.78533, 106.67029  # toa do tram -> dung lai o Buoc C de lay dung thoi tiet tai day
OUT_DIR = "../01_Data"           # thu muc se luu file ket qua cuoi cung

print(f"Tram ID {LOCATION_ID} tai toa do ({LAT}, {LON})")

# --- KHOA DU LIEU: de chay lai notebook khong lam thay doi du lieu nhom da chot ---
NGAY_DAU_CO_DINH = "2024-11-19"    # khoang thoi gian co dinh: tu ngay nay ...
NGAY_CUOI_CO_DINH = "2026-10-19"   # ... den ngay nay, du API co them ngay moi hay doi so lieu cu
GHI_DE_FILE_CHINH = False          # False: file chinh da co thi KHONG ghi de (ghi ra *_chay_lai.csv)
print("Khoang thoi gian co dinh:", NGAY_DAU_CO_DINH, "->", NGAY_CUOI_CO_DINH, "| Cho phep ghi de file chinh:", GHI_DE_FILE_CHINH)


Tram ID 3276359 tai toa do (10.78533, 106.67029)


### Khóa dữ liệu (để chạy lại không làm đổi số liệu nhóm đã chốt)

- `NGAY_DAU_CO_DINH` và `NGAY_CUOI_CO_DINH` luôn chỉ giữ các ngày trong khoảng **19/11/2024 → 19/10/2026**, dù chạy lại lúc nào (trạm vẫn đo thêm ngày mới).
- `GHI_DE_FILE_CHINH = False`: nếu `aqi_hcm_daily.csv` đã có thì **không ghi đè**, bản chạy lại được ghi ra `aqi_hcm_daily_chay_lai.csv`.

Muốn cố ý tạo lại file chính: đổi `GHI_DE_FILE_CHINH = True` (và các ngày cố định nếu cần), rồi báo cả nhóm vì mọi kết quả sẽ đổi theo.

## BƯỚC A: Xác minh trạm có thật (bằng chứng nguồn gốc)

**Bản chất:** trước khi lấy số liệu, phải chứng minh trạm mình định lấy dữ liệu **có thật và đang hoạt động** — không phải mã trạm bịa ra. Đây là một lệnh GET, tức là hỏi máy chủ: "cho tôi xem thông tin trạm số 3276359".

Kết quả trả về là **JSON** — một cấu trúc dữ liệu lồng nhau, giống một bộ hồ sơ có nhiều mục con. Trong đó có tên trạm, toạ độ, và quan trọng nhất là **danh sách sensor**: mỗi loại đo (PM2.5, nhiệt độ, độ ẩm...) có 1 sensor riêng với 1 mã ID riêng — ta cần các mã ID này để kéo dữ liệu ở bước sau.

In [4]:
def xac_minh_tram():
    # Goi GET toi dung URL cua tram nay, dinh kem key qua headers
    r = requests.get(f"https://api.openaq.org/v3/locations/{LOCATION_ID}",
                      headers=HEADERS, timeout=30)

    # status_code 200 = thanh cong. Khac 200 (vd 401/403) = key sai hoac loi
    if r.status_code != 200:
        print("LOI:", r.status_code, r.text[:200])
        print(">>> Kiem tra lai API_KEY vua dan o o tren!")
        return None   # dung ham tai day, khong chay tiep xuong duoi

    # r.json() bien chuoi JSON tra ve thanh dict/list Python dung duoc
    # ["results"] la 1 list cac tram tim thay -> [0] lay tram dau tien (va duy nhat)
    d = r.json()["results"][0]

    print("Ten tram    :", d["name"])
    print("Location ID :", d["id"])
    # "coordinates" la 1 dict con nam trong d -> phai mo 2 lop de lay duoc so
    print("Toa do      :", d["coordinates"]["latitude"], ",", d["coordinates"]["longitude"])
    # .get(key, mac_dinh) thay vi [key] -> tranh loi neu thieu muc "provider"
    print("Provider    :", d.get("provider", {}).get("name"))
    # chuoi goc dang "2024-11-19T10:30:00+07:00", [:10] cat lay 10 ky tu dau = chi ngay
    print("Du lieu tu  :", d["datetimeFirst"]["local"][:10])
    print("Den         :", d["datetimeLast"]["local"][:10])
    print(" Cac sensor co tren tram:")

    sensors = {}
    for s in d["sensors"]:                     # d["sensors"] la list, lap qua tung sensor
        ten = s["parameter"]["name"]             # ten thong so do (pm25, temperature...)
        sensors[ten] = s["id"]                    # ghi lai: ten -> ma id, de dung o buoc sau
        print(f"  - {ten:18s} id={s['id']}")       # :18s chi la can le cho dep, khong anh huong logic

    print(f">>> Kiem chung cong khai: https://explore.openaq.org/locations/{LOCATION_ID}")
    return sensors   # tra dict {ten_sensor: id} ra ngoai de dung tiep

sensors = xac_minh_tram()   # goi ham -> luc nay moi thuc su chay

Ten tram    : CMT8
Location ID : 3276359
Toa do      : 10.78533 , 106.67029
Provider    : AirGradient
Du lieu tu  : 2024-11-19
Den         : 2026-10-21
 Cac sensor co tren tram:
  - pm1                id=11357396
  - pm10               id=11357395
  - pm25               id=11357424
  - relativehumidity   id=11357398
  - temperature        id=11357401
  - um003              id=11357394
>>> Kiem chung cong khai: https://explore.openaq.org/locations/3276359


## KIỂM TRA SENSOR PM10 

Ô dưới kéo thử toàn bộ lịch sử ngày của sensor `pm10` (cùng cách với `pm25`) rồi đếm số ngày có giá trị. Kết quả: chỉ **57/657 ngày (8,7%)** có số và ngày cuối là **14/01/2025** → cảm biến PM10 đã **ngừng hoạt động**, không phải thiếu ngẫu nhiên. **Quyết định:** loại `pm10`, chỉ giữ `pm25` (biến Y), `temperature`, `relativehumidity`.

In [ ]:
def keo_sensor(sensor_id, ten):
    """Keo toan bo du lieu NGAY cua 1 sensor (co phan trang)."""
    rows = []
    for page in range(1, 10):     # goi lan luot tung trang, toi da 9 trang (thua du cho 622 ngay)
        r = requests.get(
            f"https://api.openaq.org/v3/sensors/{sensor_id}/days",
            headers=HEADERS,
            params={"limit": 1000, "page": page},   # limit=1000: xin toi da 1000 dong/lan goi
            timeout=60,
        )
        if r.status_code != 200:
            print(f"  LOI {ten}: {r.status_code}")
            break

        kq = r.json()["results"]
        if not kq:          # trang rong -> da lay het du lieu, dung vong lap
            break

        for x in kq:
            rows.append({
                "date": x["period"]["datetimeFrom"]["local"][:10],   # cat lay ngay, bo gio
                ten: x["value"],                                       # gia tri do duoc ngay do
                # % so gio thuc te do duoc trong ngay -> cang gan 100% cang dang tin
                ten + "_cov": x["coverage"]["percentComplete"],
            })

        if len(kq) < 1000:   # trang chua day 1000 dong -> day la trang cuoi, khong can goi tiep
            break

    df = pd.DataFrame(rows).drop_duplicates("date")   # gop list dict thanh bang, bo ngay trung
    print(f"  {ten:14s}: {len(df):4d} ngay  ({df['date'].min()} -> {df['date'].max()})")
    return df


# Ket qua pm25 se duoc tai su dung o Buoc B (khong keo lai lan 2, do khong ton API)
dfs = {}
dfs["pm25"] = keo_sensor(sensors["pm25"], "pm25")

if "pm10" in sensors:
    pm10_df = keo_sensor(sensors["pm10"], "pm10")   # chi de ghi vao file tho (chung minh sensor da chet), khong dung phan tich
    ty_le = 100 * pm10_df["pm10"].notna().sum() / len(dfs["pm25"])
    print(f"pm10 co du lieu {ty_le:.1f}% so ngay, ngay cuoi {pm10_df.loc[pm10_df['pm10'].notna(), 'date'].max()} -> LOAI khoi phan tich")


  pm25          :  657 ngay  (2024-11-19 -> 2026-10-19)
  pm10          :   57 ngay  (2024-11-19 -> 2025-01-14)
pm10 co du lieu 8.7% so ngay, ngay cuoi 2025-01-14 -> LOAI khoi phan tich


## BƯỚC A2: Có trạm nào khác đo các khí không?

Nhóm hỏi OpenAQ mọi trạm trong khung ±0,6° (khoảng ±65 km) quanh CMT8 và kiểm tra: còn hoạt động không, đo chất gì, phủ khoảng thời gian nào. Kết quả: **19 trạm**, trong đó:

- Chỉ **CMT8** còn hoạt động (dữ liệu đến 10/2026).
- **Không trạm nào đo NO, NO₂, CO, SO₂, O₃** (các trạm khác chỉ đo PM2.5 và vài thông số cơ bản).
- Lãnh sự quán Mỹ (ID 7440, cách 3,3 km): chỉ đo PM2.5, dữ liệu 10/11/2016–04/03/2025, đã ngừng → chồng khoảng 3,5 tháng với CMT8.
- 3 trạm AirGradient lẻ và 13 trạm HabitatMap: chỉ có từ vài ngày đến vài tuần, không đủ dùng.

**Kết luận:** các khí được lấy từ **mô hình CAMS** (Bước H), ghi rõ là số mô hình chứ không phải cảm biến. *(Muốn xem lại bảng 19 trạm: gọi `GET /v3/locations?bbox=...` của OpenAQ với khung trên.)*

## BƯỚC B: Kéo dữ liệu không khí (PM2.5, nhiệt độ, độ ẩm) — từng ngày

**Vì sao phải "phân trang" (pagination)?** API không trả hết dữ liệu trong 1 lần gọi — mỗi lần chỉ trả tối đa 1000 dòng, giống như thư viện chỉ đưa từng tập 1000 trang một, đưa hết tập này mới đưa tập kế. Vòng lặp `for page in range(1, 10)` gọi lặp lại nhiều lần, mỗi lần 1 trang, đến khi trang trả về rỗng thì dừng — nghĩa là đã lấy hết.

Chỉ lấy 3 sensor: `pm25`, `temperature`, `relativehumidity`. **Cố ý bỏ `pm10`** vì cảm biến đó đã ngừng hoạt động từ 14/01/2025 — giữ lại sẽ làm dữ liệu thiếu quá nhiều, không đủ dùng.

In [8]:
# pm25 da keo o buoc kiem tra pm10 phia tren (dfs["pm25"]) -> khong keo lai
print("Dang keo du lieu nhiet do + do am tu tram...")
can_lay_them = ["temperature", "relativehumidity"]
for ten in can_lay_them:
    if ten not in sensors:
        print(f"  CANH BAO: khong tim thay sensor {ten}")
        continue
    # API dat ten "relativehumidity" nhung minh muon luu cot ten la "humidity" cho gon
    cot = "humidity" if ten == "relativehumidity" else ten
    dfs[cot] = keo_sensor(sensors[ten], cot)

# Ghep 3 bang (pm25, temperature, humidity) lam 1, noi voi nhau qua cot chung "date"
kk = dfs["pm25"]
for ten in ["temperature", "humidity"]:
    kk = kk.merge(dfs[ten], on="date", how="outer")   # outer: giu het ngay, du co thieu o 1 ben
kk = kk.sort_values("date").reset_index(drop=True)     # sap xep lai theo thoi gian tang dan

kk.head()   # xem thu 5 dong dau de kiem tra ghep dung chua

Dang keo du lieu nhiet do + do am tu tram...
  temperature   :  657 ngay  (2024-11-19 -> 2026-10-19)
  humidity      :  657 ngay  (2024-11-19 -> 2026-10-19)


,date,pm25,pm25_cov,temperature,temperature_cov,humidity,humidity_cov
0,2024-11-19,31.2,29.0,28.8,29.0,57.9,29.0
1,2024-11-20,36.6,100.0,28.9,100.0,54.1,100.0
2,2024-11-21,42.2,100.0,28.9,100.0,53.5,100.0
3,2024-11-22,38.2,100.0,28.8,100.0,54.1,100.0
4,2024-11-23,37.1,100.0,28.5,100.0,53.7,100.0


## BƯỚC C: Kéo TOÀN BỘ dữ liệu Open-Meteo Archive 

Trạm CMT8 **không có cảm biến gió, mưa, áp suất, lớp biên khí quyển** — nó chỉ đo bụi, nhiệt độ, độ ẩm. Mà các yếu tố này lại ảnh hưởng mạnh tới bụi mịn (gió thổi bay bụi, mưa rửa trôi bụi, lớp biên thấp nhốt bụi gần mặt đất), nên phải lấy bổ sung từ nguồn khác: **Open-Meteo Archive API** — dùng đúng toạ độ của trạm CMT8, không cần API key (nguồn mở).


| Biến xin về | Mức | Dùng ở bước |
|---|---|---|
| `wind_speed_10m_max` → `wind` | ngày | D |
| `precipitation_sum` → `rain` | ngày | D và G (tính số ngày kể từ mưa) |
| `surface_pressure_mean` → `ap_suat` | ngày | E |
| `wind_direction_10m_dominant` → `wind_dir_deg` | ngày | F |
| `boundary_layer_height` → `pbl_sang_m` | **giờ** (tự gộp về ngày) | E |

Khoảng ngày xin về bắt đầu **sớm hơn ngày đầu tiên của trạm 90 ngày**: phần 90 ngày thêm này chỉ để Bước F biết "đã khô bao lâu" lúc bắt đầu, và không đưa vào bảng.


In [9]:
print("BUOC C: Keo TOAN BO du lieu Open-Meteo Archive (1 lan goi duy nhat)")

LUI_NGAY = 90          # keo som hon ngay dau tien 90 ngay, de Buoc G biet "da kho bao lau" luc bat dau
ngay_kk = pd.to_datetime(kk["date"])
tu_ngay = (ngay_kk.min() - pd.Timedelta(days=LUI_NGAY)).date()
den_ngay = min(ngay_kk.max(), pd.Timestamp(NGAY_CUOI_CO_DINH)).date()      # khong xin qua ngay co dinh

r_om = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",    # API thoi tiet, khong can key
    params={
        "latitude": LAT, "longitude": LON,               # dung DUNG toa do cua tram CMT8
        "start_date": str(tu_ngay), "end_date": str(den_ngay),
        # NGAY: gio toi da, tong mua, ap suat, huong gio chiem uu the
        "daily": "wind_speed_10m_max,precipitation_sum,surface_pressure_mean,wind_direction_10m_dominant",
        # GIO: lop bien khi quyen (chi co o muc gio, phai tu gop ve ngay o duoi)
        "hourly": "boundary_layer_height",
        "timezone": "Asia/Bangkok",
    },
    timeout=180,
)
r_om.raise_for_status()           # bao loi ngay neu API tra ma loi
d_om = r_om.json()

# --- Bang theo NGAY ---
om_ngay = pd.DataFrame({
    "date": pd.to_datetime(d_om["daily"]["time"]),
    "wind": d_om["daily"]["wind_speed_10m_max"],
    "rain": d_om["daily"]["precipitation_sum"],
    "ap_suat": d_om["daily"]["surface_pressure_mean"],
    "wind_dir_deg": d_om["daily"]["wind_direction_10m_dominant"],
})

# --- Lop bien: du lieu theo GIO -> lay trung binh khung 0h-6h sang (luc nghich nhiet manh nhat trong ngay) ---
h_om = pd.DataFrame({
    "dt": pd.to_datetime(d_om["hourly"]["time"]),
    "pbl": d_om["hourly"]["boundary_layer_height"],
})
h_om["ngay"] = h_om["dt"].dt.date
h_om["gio"] = h_om["dt"].dt.hour
pbl_sang = h_om[h_om["gio"].between(0, 6)].groupby("ngay", as_index=False)["pbl"].mean()
pbl_sang.columns = ["date", "pbl_sang_m"]
pbl_sang["date"] = pd.to_datetime(pbl_sang["date"])
om_ngay = om_ngay.merge(pbl_sang, on="date", how="left")

# --- Bang thoi tiet cho Buoc D: chi lay dung khoang ngay ma tram co du lieu (bo 90 ngay lui) ---
tt = om_ngay[["date", "wind", "rain"]].copy()
tt["date"] = tt["date"].dt.strftime("%Y-%m-%d")                  # cung dang chuoi voi kk["date"] de ghep o Buoc D
tt = tt[(tt["date"] >= str(ngay_kk.min().date())) & (tt["date"] <= str(den_ngay))].reset_index(drop=True)

print(f"  Open-Meteo: {len(om_ngay)} ngay ({om_ngay['date'].min().date()} -> {om_ngay['date'].max().date()}), "
      f"trong do {len(tt)} ngay dung cho bang chinh, {len(om_ngay) - len(tt)} ngay lui lam nen")
tt.head()


  Thoi tiet:  700 ngay (2024-11-19 -> 2026-10-19)


,date,wind,rain
0,2024-11-19,8.9,0.9
1,2024-11-20,13.0,1.4
2,2024-11-21,8.7,6.0
3,2024-11-22,8.9,0.4
4,2024-11-23,10.0,0.3


## BƯỚC D: Ghép 2 nguồn + tạo biến phân tích

**Ghép (merge/join)** = giống ghép 2 bảng Excel theo 1 cột chung — ở đây là cột `date`. Dùng `how="inner"` nghĩa là chỉ giữ những ngày **có mặt ở cả 2 bảng**, ngày nào chỉ có ở 1 bảng thì bỏ (để không có ô trống).

Sau khi ghép, tạo thêm các biến:
- `month`, `dow` (thứ trong tuần) — dùng cho kiểm định ANOVA theo tháng, theo ngày
- `mua` (mùa mưa: tháng 5–10 / mùa khô: còn lại) — dùng cho t-test 2 mùa. Đây là quy ước của nhóm, căn cứ và giới hạn nêu ở bảng nguồn đầu notebook
- Các biến phân loại ô nhiễm (`vn_aqi_cat`, `kem_tro_len`…) **không** tạo ở đây mà ở Bước I, theo **chuẩn do Việt Nam công bố** (VN_AQI, QĐ 1459/QĐ-TCMT). Nhóm không dùng thang Mỹ (US EPA) nữa để không lẫn hai chuẩn.

**Hai loại "thiếu" khác nhau, đừng nhầm:**
- **Dòng có ô trống** (ví dụ trạm có dòng ngày đó nhưng thiếu một số đo): `dropna` bỏ các dòng này. Ô bên dưới in ra "bỏ N dòng thiếu dữ liệu".
- **Ngày vắng mặt hẳn khỏi bảng** (trạm mất dữ liệu, không có dòng nào cho ngày đó): `dropna` **không** thấy chúng vì chúng không tồn tại trong bảng. Ô bên dưới đếm riêng trên lịch liên tục và in ra các khoảng vắng. Ở dữ liệu của nhóm có **43 ngày** như vậy, dù "bỏ 0 dòng thiếu dữ liệu".


In [10]:
# Ghep khong khi (kk) voi thoi tiet (tt) theo cot date
# how="inner": CHI giu ngay co mat o CA HAI bang -> khong con o trong
df = kk.merge(tt, on="date", how="inner")
df["date"] = pd.to_datetime(df["date"])   # doi cot date tu chuoi text -> kieu ngay-thang that

# KHOA KHOANG THOI GIAN: chi giu cac ngay tu NGAY_DAU_CO_DINH den NGAY_CUOI_CO_DINH
n_truoc_khoa = len(df)
df = df[(df["date"] >= pd.Timestamp(NGAY_DAU_CO_DINH)) & (df["date"] <= pd.Timestamp(NGAY_CUOI_CO_DINH))].copy()
print(f"Khoa khoang {NGAY_DAU_CO_DINH} -> {NGAY_CUOI_CO_DINH}: bo {n_truoc_khoa - len(df)} ngay nam ngoai khoang")
df["month"] = df["date"].dt.month          # trich thang (1-12) -> dung cho ANOVA theo thang
df["dow"] = df["date"].dt.dayofweek         # trich thu trong tuan, 0 = Thu Hai

# Mua mua Nam Bo: thang 5 -> 10 la QUY UOC cua nhom (nguon va gioi han: xem bang nguon dau notebook)
df["mua"] = np.where(df["month"].isin([5, 6, 7, 8, 9, 10]), "Mua mua", "Mua kho")

# Cac bien phan loai o nhiem (vn_aqi_cat, kem_tro_len...) duoc tao o BUOC I theo chuan Viet Nam

truoc = len(df)
# dropna: bo nhung dong con thieu bat ky bien nao trong 5 bien chinh nay
df = df.dropna(subset=["pm25", "temperature", "humidity", "wind", "rain"])
print(f"Truoc lam sach: {truoc} dong")
print(f"Sau lam sach  : {len(df)} dong (bo {truoc - len(df)} dong thieu du lieu)")

# CHU Y: dropna o tren chi bo cac DONG CO O TRONG. Nhung ngay VANG MAT han khoi bang (tram mat du lieu,
# khong co dong nao) khong bi dem o day -> phai dem rieng tren lich lien tuc tu ngay dau den ngay cuoi.
lich_day_du = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
ngay_vang = pd.Series(lich_day_du.difference(df["date"]))
print()
print(f"Ngay VANG MAT khoi bang (khong co dong nao): {len(ngay_vang)} ngay")
if len(ngay_vang):
    so_khoang = (ngay_vang.diff() != pd.Timedelta(days=1)).cumsum()      # chia thanh cac khoang lien tiep
    for _, kh in ngay_vang.groupby(so_khoang):
        print(f"  {kh.iloc[0].date()} -> {kh.iloc[-1].date()}  ({len(kh)} ngay)")

df.describe()   # xem nhanh thong ke tong quat -> kiem tra so co hop ly khong (vd am, qua lon...)

Truoc lam sach: 657 dong
Sau lam sach  : 657 dong (bo 0 dong thieu du lieu)


,date,pm25,pm25_cov,temperature,temperature_cov,humidity,humidity_cov,wind,rain,month,dow
count,657,657.000000,657.000000,657.000000,657.000000,657.000000,657.000000,657.00000,657.000000,657.000000,657.000000
mean,2025-11-18 14:54:14.794520,32.377473,97.936073,29.054947,97.936073,55.130746,97.936073,13.31035,6.119787,6.570776,2.995434
min,2024-11-19 00:00:00,11.300000,4.000000,26.200000,4.000000,38.500000,4.000000,4.40000,0.000000,1.000000,0.000000
25%,2025-05-30 00:00:00,24.800000,100.000000,28.100000,100.000000,51.200000,100.000000,10.70000,0.200000,4.000000,1.000000
50%,2025-11-25 00:00:00,30.300000,100.000000,29.000000,100.000000,55.200000,100.000000,13.00000,2.200000,7.000000,3.000000
75%,2026-05-08 00:00:00,38.900000,100.000000,29.900000,100.000000,59.500000,100.000000,15.50000,9.500000,9.000000,5.000000
max,2026-10-19 00:00:00,92.800000,100.000000,34.100000,100.000000,72.400000,100.000000,23.40000,45.700000,12.000000,6.000000
std,NaN,11.286558,10.458200,1.260956,10.458200,6.263855,10.458200,3.29826,7.980418,3.363850,1.999614


## BƯỚC E: Ghép lớp biên khí quyển, áp suất và hướng gió (đã kéo ở Bước C, không gọi API nữa)

### E1. Lớp biên khí quyển + áp suất

- **Lớp biên khí quyển** là chiều cao lớp không khí xáo trộn gần mặt đất. Lớp càng thấp, bụi càng bị nhốt gần mặt đất → PM2.5 càng cao (cơ chế nghịch nhiệt). Chỉ có theo giờ nên lấy trung bình **0h–6h sáng**, lúc nghịch nhiệt mạnh nhất.
- **Áp suất mặt đất:** áp suất cao thường đi cùng không khí tù đọng, bụi khó phát tán.
- Đã thử: thêm 2 biến này vào mô hình 4 biến, Adjusted R² tăng 0,293 → **0,350**, VIF < 2,2, cả hai có ý nghĩa (p < 0,001).

### E2. Hướng gió

Bụi bay từ đâu tới quan trọng không kém tốc độ gió: gió từ phía khu công nghiệp/đường cao tốc/nơi đốt rơm rạ mang bụi về, gió từ biển pha loãng bụi. `wind_direction_10m_dominant` là hướng gió **thổi từ đâu tới** (0° = Bắc, 90° = Đông, 180° = Nam, 270° = Tây). Hướng là biến **vòng tròn** (359° và 1° gần nhau) nên không đưa thẳng số độ vào hồi quy mà đổi sang `sin`, `cos`.

| Cột mới | Ý nghĩa |
|---|---|
| `wind_dir_deg` | hướng gió chiếm ưu thế trong ngày, độ (0–360) |
| `wind_sin` | gần +1: gió từ Đông; gần −1: từ Tây |
| `wind_cos` | gần +1: gió từ Bắc; gần −1: từ Nam |
| `wind_sector` | 8 hướng chữ (Bac, Dong Bac, …) để dễ đọc và vẽ |

In [12]:
print("BUOC E: Ghep lop bien khi quyen, ap suat va huong gio (da keo o Buoc C)")

# Ghep vao bang chinh theo ngay (left: giu nguyen cac dong dang co)
df = df.merge(om_ngay[["date", "pbl_sang_m", "ap_suat"]], on="date", how="left")

thieu_f = df[["pbl_sang_m", "ap_suat"]].isna().sum().sum()
print(f"So o thieu sau khi ghep: {thieu_f}")
df = df.dropna(subset=["pbl_sang_m", "ap_suat"])
print(f"Con lai: {len(df)} dong, {df.shape[1]} cot")

# --- E2. Huong gio ---
# Ghep vao bang chinh theo ngay (left: giu nguyen cac dong dang co)
df = df.merge(om_ngay[["date", "wind_dir_deg"]], on="date", how="left")
assert df["wind_dir_deg"].notna().all(), "Co ngay thieu huong gio - kiem tra lai API"

# Doi sang toa do tren vong tron: 359 do va 1 do phai gan nhau
rad = np.deg2rad(df["wind_dir_deg"])                 # do -> radian
df["wind_sin"] = np.sin(rad)                         # thanh phan Dong(+) - Tay(-)
df["wind_cos"] = np.cos(rad)                         # thanh phan Bac(+) - Nam(-)

# 8 huong chu: moi huong rong 45 do, Bac nam giua 337.5 -> 22.5 nen cong 22.5 truoc khi chia 45
ten_huong = ["Bac", "Dong Bac", "Dong", "Dong Nam", "Nam", "Tay Nam", "Tay", "Tay Bac"]
chi_so = (((df["wind_dir_deg"] + 22.5) % 360) // 45).astype(int)
df["wind_sector"] = chi_so.map(dict(enumerate(ten_huong)))

print("So dong:", len(df), "| huong gio thieu:", int(df["wind_dir_deg"].isna().sum()))
print("\nSo ngay theo huong gio:")
print(df["wind_sector"].value_counts().reindex(ten_huong).fillna(0).astype(int))
print("\nPM2.5 trung binh theo huong gio (ug/m3):")
print(df.groupby("wind_sector")["pm25"].agg(["mean", "count"]).reindex(ten_huong).round(1))
df[["date", "wind", "wind_dir_deg", "wind_sin", "wind_cos", "wind_sector"]].head()

BUOC F: Lay them Lop bien khi quyen + Ap suat (Open-Meteo)
So o thieu sau khi ghep: 0
Con lai: 657 dong, 14 cot


,date,pm25,pbl_sang_m,ap_suat
0,2024-11-19,31.2,110.714286,1008.6
1,2024-11-20,36.6,82.142857,1008.3
2,2024-11-21,42.2,137.142857,1007.7
3,2024-11-22,38.2,80.714286,1007.9
4,2024-11-23,37.1,100.000000,1008.5


**Cách đọc:** hướng nào có PM2.5 trung bình cao hơn hẳn thì đó là hướng "mang bụi về". Chú ý cột `count`: hướng chỉ có vài ngày thì trung bình không đáng tin. Hướng gió đổi theo **mùa gió** (Đông Bắc mùa khô, Tây Nam mùa mưa) nên khi phân tích phải tách khỏi hiệu ứng mùa.

## BƯỚC F: Số ngày kể từ cơn mưa gần nhất (`ngay_tu_mua`)

Mưa rửa bụi không chỉ trong ngày mưa: sau nhiều ngày khô, bụi tích tụ dần. Cột `rain` chỉ cho biết **hôm nay** có mưa không, còn `ngay_tu_mua` cho biết **đã bao lâu** chưa mưa.

**Định nghĩa:** số ngày tính từ ngày gần nhất có mưa **≥ 1 mm** (ngày mưa = 0, hôm sau không mưa = 1, rồi 2, 3…). Ngưỡng 1 mm vì vài phần mười mm chỉ là mưa phùn, không đủ rửa bụi.

**Hai chỗ dễ sai:**
1. Phải có mưa của **90 ngày trước** ngày đầu tiên (đã kéo ở Bước C, chỉ làm nền, không đưa vào bảng) mới biết 19/11/2024 đã khô bao lâu.
2. Bảng đang thiếu ngày, nên đếm trên **lịch đầy đủ từng ngày liên tục**: ngày bảng thiếu thì lấy mưa từ API, ngày bảng có thì dùng đúng cột `rain` đã lưu (để khớp với cột cả nhóm đã dùng).

In [14]:
print("BUOC F: Tinh so ngay ke tu con mua gan nhat")

NGUONG_MUA = 1.0       # mm: mua tu 1 mm tro len moi tinh la "co mua" (duoi 1 mm la mua phun)
LUI_NGAY = 90          # 90 ngay lui da keo san o Buoc C de biet "da kho bao lau" luc bat dau

d_dau = df["date"].min() - pd.Timedelta(days=LUI_NGAY)
d_cuoi = df["date"].max()

# Luong mua theo ngay, ke ca 90 ngay lui, lay tu bang da keo o Buoc C (khong goi API nua)
mua_api = om_ngay.set_index("date")["rain"].astype(float)

# Lich DAY DU tung ngay, khong bo sot ngay nao (ke ca ngay bang dang thieu)
lich = pd.date_range(d_dau, d_cuoi, freq="D")
mua_ngay = mua_api.reindex(lich)

# Ngay nao bang da co: dung cot rain da luu (khop voi cac cot cu); ngay thieu thi giu so cua API
mua_luu = df.set_index("date")["rain"]
mua_ngay.loc[mua_luu.index] = mua_luu
assert mua_ngay.notna().all(), "Co ngay khong co du lieu mua - kiem tra lai API"

n_thieu = len(lich) - LUI_NGAY - len(df)
print(f"Lich day du: {len(lich) - LUI_NGAY} ngay | bang dang co: {len(df)} dong | ngay bang con thieu: {n_thieu}")

# Dem so ngay ke tu lan mua >= 1 mm gan nhat
dem, ket_qua = 0, []
for luong_mua in mua_ngay.values:
    dem = 0 if luong_mua >= NGUONG_MUA else dem + 1
    ket_qua.append(dem)
ngay_tu_mua = pd.Series(ket_qua, index=lich)

df["ngay_tu_mua"] = df["date"].map(ngay_tu_mua)       # gan lai vao dung ngay cua bang chinh

print("\nPhan bo ngay_tu_mua (so ngay):")
print(df["ngay_tu_mua"].describe().round(1))
print("\nPM2.5 trung binh theo so ngay khong mua (ug/m3):")
nhom = pd.cut(df["ngay_tu_mua"], bins=[-1, 0, 2, 6, 13, 1000], labels=["0 (co mua hom nay)", "1-2", "3-6", "7-13", ">= 14"])
print(df.groupby(nhom, observed=True)["pm25"].agg(["mean", "count"]).round(1))
df[["date", "rain", "ngay_tu_mua"]].head(10)

BUOC H: Tinh so ngay ke tu con mua gan nhat
Lich day du: 700 ngay | bang dang co: 657 dong | ngay bang con thieu: 43

Phan bo ngay_tu_mua (so ngay):
count    657.0
mean       2.8
std        6.3
min        0.0
25%        0.0
50%        0.0
75%        2.0
max       40.0
Name: ngay_tu_mua, dtype: float64

PM2.5 trung binh theo so ngay khong mua (ug/m3):
                    mean  count
ngay_tu_mua                    
0 (co mua hom nay)  30.7    405
1-2                 31.9     95
3-6                 38.2     66
7-13                38.4     53
>= 14               32.8     38


,date,rain,ngay_tu_mua
0,2024-11-19,0.9,4
1,2024-11-20,1.4,0
2,2024-11-21,6.0,0
3,2024-11-22,0.4,1
4,2024-11-23,0.3,2
5,2024-11-24,3.8,0
6,2024-11-25,0.1,1
7,2024-11-26,0.0,2
8,2024-11-27,0.9,3
9,2024-11-28,0.7,4


**Cách đọc:** nếu PM2.5 trung bình tăng dần theo số ngày không mưa thì bụi thật sự tích tụ (dùng được cho cảnh báo). Nhưng ngày khô lâu hầu hết rơi vào mùa khô, nên phải kiểm tra lại khi đã tính tháng/mùa (notebook 03), không kết luận vội.

## BƯỚC G: Thêm NGÀY NGHỈ LỄ & TẾT

**Vì sao cần:** dịp Tết, người nhập cư về quê, xe cộ giảm, công trình nghỉ (nhưng có thể tăng đốt vàng mã, pháo) → PM2.5 khác ngày thường. Đây là cách "đo" tác động hoạt động con người lên bụi mịn khi thời tiết gần như không đổi.

**Định nghĩa:** `ky_nghi_le` = 1 nếu ngày nằm trong **kỳ nghỉ lễ chính thức** của cơ quan nhà nước (tính cả cuối tuần nối liền kỳ nghỉ); `tet` = 1 nếu thuộc kỳ nghỉ **Tết Nguyên đán** (tập con của `ky_nghi_le`). Cuối tuần thông thường không tính (đã có `dow`).

**Ngày nghỉ mỗi năm một khác (nghỉ bù, hoán đổi) nên phải ghi tay đúng văn bản công bố:**

| Kỳ nghỉ | Ngày nghỉ liền mạch | Nguồn |
|---|---|---|
| Tết Dương lịch 2025 | 01/01/2025 | Bộ luật Lao động 2019, Điều 112 ([bài tổng hợp: VnEconomy](https://vneconomy.vn/nguoi-lao-dong-co-22-ngay-nghi-le-tet-trong-nam-2025.htm)) |
| **Tết Ất Tỵ 2025** | 25/01 → 02/02/2025 (9 ngày) | [VOH](https://voh.com.vn/xa-hoi/bo-noi-vu-thong-nhat-nghi-tet-at-ty-trong-9-ngay-552590.html), [SGGP](https://www.sggp.org.vn/tet-nguyen-dan-at-ty-2025-thu-tuong-dong-y-nghi-9-ngay-lien-tuc-post770158.html) |
| Giỗ Tổ Hùng Vương 2025 (10/3 Âm lịch = Thứ Hai 07/04) | 05/04 → 07/04/2025 | Bộ luật Lao động 2019, Điều 112 |
| Giải phóng miền Nam + Quốc tế Lao động 2025 | 30/04 → 04/05/2025 | Các báo điện tử đăng lại thông báo của Bộ Nội vụ (VnEconomy, VOH…): nghỉ 30/04–04/05, làm bù thứ Bảy 26/04 |
| Quốc khánh 2025 | 30/08 → 02/09/2025 | Các báo điện tử đăng lại phương án Thủ tướng đồng ý (nghỉ 4 ngày) |
| Tết Dương lịch 2026 | 01/01/2026 | Bộ luật Lao động 2019, Điều 112 ([Thư Viện Pháp Luật](https://thuvienphapluat.vn/hoi-dap-phap-luat/tet-duong-lich-2026-duoc-nghi-1-ngay-hay-4-ngay-138075149.html)); có báo nói nghỉ 4 ngày 01–04/01, chưa tìm được văn bản xác nhận nên **không** tính |
| **Tết Bính Ngọ 2026** | 14/02 → 22/02/2026 (9 ngày) | Thông báo 9441/TB-BNV ngày 16/10/2025 của Bộ Nội vụ ([Luật Việt Nam](https://luatvietnam.vn/tin-van-ban-moi/lich-nghi-tet-nguyen-dan-nam-2026-cho-can-bo-cong-chuc-vien-chuc-va-nld-186-104767-article.html)) |
| Giỗ Tổ Hùng Vương 2026 (10/3 Âm lịch = Chủ Nhật 26/04) | 25/04 → 27/04/2026 | [Báo Văn hóa](https://baovanhoa.vn/doi-song/chi-tiet-lich-nghi-gio-to-hung-vuong-304-15-va-quoc-khanh-2026-216549.html) |
| 30/04 + 01/05/2026 | 30/04 → 03/05/2026 | [PLO](https://plo.vn/chi-tiet-lich-nghi-le-30-4-va-gio-to-hung-vuong-2026-tai-tphcm-post902875.html) |
| Quốc khánh 2026 | 29/08 → 02/09/2026 | Thông báo 9441/TB-BNV (Bộ Nội vụ), [CafeF](https://cafef.vn/chinh-thuc-bo-noi-vu-thong-bao-quoc-khanh-2-9-nam-nay-duoc-nghi-5-ngay-lien-tuc-188260728185305639.chn) |

**Giới hạn cần nói thật trong báo cáo:** đây là lịch của **khối cơ quan nhà nước**. Doanh nghiệp tư nhân có thể nghỉ khác (nhiều nơi nghỉ Tết dài hơn, một số nơi ít hơn). Một số ngày ở bảng trên lấy từ báo điện tử đăng lại thông báo, không phải văn bản gốc; nếu thầy hỏi, nói rõ điều này.

In [15]:
print("BUOC G: Gan co ky nghi le va Tet")

# (tu_ngay, den_ngay, la_Tet?, ten) - chep tu bang nguon o tren
KY_NGHI = [
    ("2025-01-01", "2025-01-01", 0, "Tet Duong lich 2025"),
    ("2025-01-25", "2025-02-02", 1, "Tet Nguyen dan At Ty 2025"),
    ("2025-04-05", "2025-04-07", 0, "Gio To Hung Vuong 2025"),
    ("2025-04-30", "2025-05-04", 0, "30/4 - 1/5 nam 2025"),
    ("2025-08-30", "2025-09-02", 0, "Quoc khanh 2025"),
    ("2026-01-01", "2026-01-01", 0, "Tet Duong lich 2026"),
    ("2026-02-14", "2026-02-22", 1, "Tet Nguyen dan Binh Ngo 2026"),
    ("2026-04-25", "2026-04-27", 0, "Gio To Hung Vuong 2026"),
    ("2026-04-30", "2026-05-03", 0, "30/4 - 1/5 nam 2026"),
    ("2026-08-29", "2026-09-02", 0, "Quoc khanh 2026"),
]
DANH_SACH_DEN = pd.Timestamp("2026-11-23")      # lich tren chi day du den het ngay nay (24/11/2026 la ngay nghi moi: Ngay Van hoa VN)

if df["date"].max() > DANH_SACH_DEN:
    print("CANH BAO: du lieu da qua", DANH_SACH_DEN.date(), "- hay bo sung ky nghi moi vao KY_NGHI roi chay lai.")

df["ky_nghi_le"] = 0
df["tet"] = 0
for tu, den, la_tet, ten in KY_NGHI:
    trong_ky = (df["date"] >= tu) & (df["date"] <= den)      # dieu kien loc cac ngay thuoc ky nghi
    df.loc[trong_ky, "ky_nghi_le"] = 1
    if la_tet:
        df.loc[trong_ky, "tet"] = 1
    print(f"{ten:32s} {tu} -> {den} | so dong trong bang: {int(trong_ky.sum())}")

print("\nTong so ngay nghi le:", int(df["ky_nghi_le"].sum()), "| trong do Tet:", int(df["tet"].sum()))
print("\nPM2.5 trung binh (ug/m3): ngay thuong vs ky nghi le vs Tet")
print(df.groupby(["ky_nghi_le", "tet"])["pm25"].agg(["mean", "count"]).round(1))

BUOC I: Gan co ky nghi le va Tet
Tet Duong lich 2025              2025-01-01 -> 2025-01-01 | so dong trong bang: 1
Tet Nguyen dan At Ty 2025        2025-01-25 -> 2025-02-02 | so dong trong bang: 0
Gio To Hung Vuong 2025           2025-04-05 -> 2025-04-07 | so dong trong bang: 3
30/4 - 1/5 nam 2025              2025-04-30 -> 2025-05-04 | so dong trong bang: 5
Quoc khanh 2025                  2025-08-30 -> 2025-09-02 | so dong trong bang: 4
Tet Duong lich 2026              2026-01-01 -> 2026-01-01 | so dong trong bang: 1
Tet Nguyen dan Binh Ngo 2026     2026-02-14 -> 2026-02-22 | so dong trong bang: 9
Gio To Hung Vuong 2026           2026-04-25 -> 2026-04-27 | so dong trong bang: 3
30/4 - 1/5 nam 2026              2026-04-30 -> 2026-05-03 | so dong trong bang: 4
Quoc khanh 2026                  2026-08-29 -> 2026-09-02 | so dong trong bang: 5

Tong so ngay nghi le: 35 | trong do Tet: 9

PM2.5 trung binh (ug/m3): ngay thuong vs ky nghi le vs Tet
                mean  count
ky_nghi_le tet 

**Hai cạm bẫy khi đọc:**
1. Ngày nghỉ rất ít (~35 ngày trong gần 2 năm). **Tết Ất Tỵ 2025 rơi đúng vào khoảng trạm mất dữ liệu** (43 ngày vắng), nên chỉ còn **một** kỳ Tết (2026, 9 ngày) để so sánh. Với cỡ mẫu này, trung bình thấp hơn chỉ là **gợi ý**, không phải kết luận chắc.
2. Tết rơi vào **cuối mùa khô** (tháng 1–2), vốn nhiều bụi. Phải so Tết với **các ngày thường cùng tháng 1–2**, đừng so với trung bình cả năm kẻo lẫn hiệu ứng mùa vào hiệu ứng ngày lễ.

## BƯỚC H: Thêm các khí (CO, NO, NO₂, SO₂, O₃) và PM10 — từ mô hình CAMS qua Open-Meteo

PM2.5 không đứng một mình: các khí đi cùng nó **cho biết bụi nghiêng về nguồn nào**. Đây là phần "nghiệp vụ môi trường" thầy yêu cầu.

| Chất | Chủ yếu đến từ | Đọc kết quả như thế nào |
|---|---|---|
| **NO, NO₂** | khí thải xe cộ, động cơ đốt trong | cùng tăng với PM2.5 → bụi nghiêng về **giao thông** |
| **CO** | cháy không hoàn toàn: xe máy, đốt rác, đốt sinh khối | cùng tăng → **đốt cháy** nói chung |
| **SO₂** | đốt nhiên liệu hóa thạch: nhà máy điện, công nghiệp, cả xe dùng nhiên liệu nhiều lưu huỳnh | cùng tăng → **đốt nhiên liệu hóa thạch** nói chung (xe máy góp khoảng 38% SO₂ ở TP.HCM theo T13) |
| **O₃** | không thải trực tiếp: hình thành khi NOx gặp ánh nắng | cao vào ngày nắng; thường **ngược** với bụi giao thông (NO bị O₃ tiêu hao) |
| **PM10** | bụi thô: bụi đường, xây dựng | PM2.5/PM10 gần 1 → bụi mịn từ đốt cháy; thấp → bụi thô |

*Tài liệu cho cột "Chủ yếu đến từ": T6 (NO₂), T7 (CO), T8 (SO₂), T9 (O₃), T13 (tỷ trọng xe máy); link ở bảng tài liệu đầu notebook.*

**Hai điều phải nói thật:**
1. **Đây là dữ liệu MÔ HÌNH, không phải cảm biến.** OpenAQ không có trạm nào đo các khí này quanh TP.HCM (đã tra ở Bước A2). Open-Meteo lấy từ mô hình **CAMS** (Copernicus, Liên minh châu Âu), giá trị trên **ô lưới thô** (~45 km, theo [tài liệu Open-Meteo](https://open-meteo.com/en/docs/air-quality-api)). Cột có hậu tố `_cams` để không ai nhầm với số đo thật.
2. **"Cùng tăng với" không phải "gây ra".** Các khí là chỉ báo cùng nguồn, không phải nguyên nhân; đưa vào hồi quy sẽ có đa cộng tuyến (NO, NO₂, CO thường tăng giảm cùng nhau).

**Đơn vị thời gian:** trả theo giờ, lấy **trung bình theo ngày** (múi giờ Việt Nam); riêng O₃ lấy thêm **giá trị giờ cao nhất trong ngày** vì O₃ chỉ cao buổi trưa nắng.

**Kiểm chứng độ tin cậy:** so PM2.5 của chính mô hình CAMS với `pm25` của trạm CMT8 — làm ở **notebook 01, Bằng chứng 4** (r = 0,31; mô hình lệch theo mùa), nên các khí chỉ dùng để xem xu hướng chung.

In [16]:
print("BUOC H: Lay cac khi tu mo hinh CAMS (Open-Meteo Air Quality)")

KHI = {                                  # ten cua Open-Meteo -> ten cot cua minh
    "carbon_monoxide": "co_cams",
    "nitrogen_monoxide": "no_cams",
    "nitrogen_dioxide": "no2_cams",
    "sulphur_dioxide": "so2_cams",
    "ozone": "o3_cams",
    "pm10": "pm10_cams",
}

r_j = requests.get(
    "https://air-quality-api.open-meteo.com/v1/air-quality",
    params={
        "latitude": LAT, "longitude": LON,
        "start_date": str(df["date"].min().date()),
        "end_date": str(df["date"].max().date()),
        "hourly": ",".join(KHI.keys()),       # cac chat nay chi co o muc GIO
        "timezone": "Asia/Bangkok",
    },
    timeout=180,
)
r_j.raise_for_status()
d_j = r_j.json()

gio = pd.DataFrame(d_j["hourly"])
gio["time"] = pd.to_datetime(gio["time"])
gio["date"] = gio["time"].dt.normalize()            # cat gio, giu ngay
gio = gio.rename(columns=KHI)

# Gop gio -> ngay: trung binh moi chat; rieng O3 lay them gia tri gio cao nhat trong ngay
khi_ngay = gio.groupby("date", as_index=False)[list(KHI.values())].mean()
khi_ngay["o3_max_cams"] = gio.groupby("date")["o3_cams"].max().values

df = df.merge(khi_ngay, on="date", how="left")
cot_khi = list(KHI.values()) + ["o3_max_cams"]

print(f"So ngay co khi: {df['co_cams'].notna().sum()} / {len(df)}")
print("\nThong ke (ug/m3, trung binh theo ngay, MO HINH CAMS):")
print(df[cot_khi].describe().loc[["mean", "min", "max"]].round(1).T)


BUOC H: Lay cac khi tu mo hinh CAMS (Open-Meteo Air Quality)
So ngay co khi: 657 / 657

Thong ke (ug/m3, trung binh theo ngay, MO HINH CAMS):
              mean    min     max
co_cams      880.2  371.5  2252.4
no_cams        3.7    0.5    16.1
no2_cams      35.1   14.8    72.8
so2_cams      20.3    4.3    55.7
o3_cams       57.0   16.1   119.9
pm10_cams     31.9   10.7    73.1
o3_max_cams  130.1   43.0   241.0


## BƯỚC I: Gắn THANG ĐO CHÍNH THỨC CỦA VIỆT NAM

Toàn bộ biến phân loại ô nhiễm (mức, ngưỡng cảnh báo, biến Y 0/1) đều tính theo chuẩn Việt Nam ở bước này.

| Chuẩn | Văn bản (nguồn) | Giá trị PM2.5 dùng ở đây |
|---|---|---|
| **VN_AQI** — chỉ số chất lượng không khí chính thức, dùng để công bố và khuyến cáo sức khỏe | **Quyết định 1459/QĐ-TCMT** ngày 12/11/2019 của Tổng cục Môi trường, **Bảng 2** (giá trị BPi). [Văn bản gốc](https://cem.gov.vn/storage/news_file_attach/QD%201459%20TCMT%20ngay%2012.11.2019%20AQI.pdf) | PM2.5 trung bình 24 giờ: Tốt 0–25; Trung bình 25–50; Kém 50–80; Xấu 80–150; Rất xấu 150–250; Nguy hại > 250 (µg/m³) |
| **QCVN 05:2023/BTNMT** — quy chuẩn kỹ thuật quốc gia về chất lượng không khí xung quanh | Ban hành kèm **Thông tư 01/2023/TT-BTNMT** ngày 13/03/2023, **Bảng 1**. [Văn bản gốc](https://thuvienmoitruong.vn/wp-content/uploads/2025/05/QCVN05_2023_BTNMT_919891.pdf) | PM2.5 trung bình 24 giờ ≤ **50** µg/m³, **riêng từ 01/01/2026 là 45** µg/m³ (chú thích (*) của Bảng 1); trung bình năm ≤ 25 |
| **WHO 2021** — hướng dẫn toàn cầu (khuyến nghị, không bắt buộc) | [WHO Global Air Quality Guidelines (2021)](https://www.who.int/publications/i/item/9789240034228) | PM2.5 trung bình 24 giờ 15, trung bình năm 5 µg/m³ |

**Lấy gì và dùng làm gì (từng văn bản):**
- **QĐ 1459/QĐ-TCMT:** lấy các **mốc nồng độ PM2.5 24 giờ** và công thức nội suy → dùng tính chỉ số `vn_aqi_pm25`, nhãn mức `vn_aqi_cat`, và biến Y `kem_tro_len` (từ mức Kém, nhóm nhạy cảm bắt đầu gặp vấn đề sức khỏe theo Bảng 4–5).
- **QCVN 05:2023/BTNMT:** lấy **giới hạn PM2.5 24 giờ** (50, và 45 từ 01/01/2026) → dùng tính `qcvn_24h` và `vuot_qcvn`, để báo cáo mức tuân thủ quy chuẩn.
- **WHO 2021:** lấy **khuyến nghị 15 µg/m³ (24 giờ)** → dùng tính `vuot_who` để đối chiếu với chuẩn quốc tế. Giá trị 15 và 5 đã đối chiếu với bản PDF gốc của WHO (Bảng 0.1, trang 19 của tệp).

**Cách tính VN_AQI (Công thức 2 của quyết định):** nồng độ $C$ nằm giữa hai mốc $BP_i$ và $BP_{i+1}$ thì AQI được **nội suy tuyến tính** giữa $I_i$ và $I_{i+1}$:

$$AQI=\frac{I_{i+1}-I_i}{BP_{i+1}-BP_i}\,(C-BP_i)+I_i$$

Ví dụ: PM2.5 = 37.5 nằm đúng giữa mốc 25 (AQI 50) và mốc 50 (AQI 100) nên AQI = 75. Kết quả **làm tròn thành số nguyên**.

**Các cột mới:**

| Cột | Ý nghĩa |
|---|---|
| `vn_aqi_pm25` | chỉ số thành phần PM2.5 của VN_AQI (0–500) |
| `vn_aqi_cat` | mức: Tot / Trung binh / Kem / Xau / Rat xau / Nguy hai |
| `qcvn_24h` | giới hạn 24 giờ của QCVN áp dụng cho ngày đó: 50, hoặc 45 từ 01/01/2026 |
| `vuot_qcvn` | 1 nếu PM2.5 vượt giới hạn QCVN của ngày đó |
| `vuot_who` | 1 nếu PM2.5 > 15 (WHO 2021) |
| `kem_tro_len` | 1 nếu mức từ **Kém trở lên** (AQI > 100, tức PM2.5 > 50): ngưỡng bắt đầu cảnh báo sức khỏe cho nhóm nhạy cảm |

**Lưu ý quan trọng:** VN_AQI chính thức là **giá trị lớn nhất** trong các chỉ số thành phần (PM2.5, PM10, O3, NO2, SO2, CO). Trạm CMT8 chỉ có PM2.5 (PM10 hỏng từ 14/01/2025), nên `vn_aqi_pm25` là **chỉ số thành phần PM2.5**: nó là cận dưới của VN_AQI thật, không phải VN_AQI đầy đủ.

In [17]:
print("BUOC I: Gan thang do chinh thuc cua Viet Nam")

# Bang 2, QD 1459/QD-TCMT (12/11/2019): cot PM2.5 (trung binh 24 gio), don vi ug/m3
BP_PM25 = [0, 25, 50, 80, 150, 250, 350, 500]     # BPi: nong do PM2.5 tai cac moc
I_AQI   = [0, 50, 100, 150, 200, 300, 400, 500]   # Ii : gia tri AQI tuong ung voi moi moc

def vn_aqi_pm25(c):
    # np.interp = noi suy tuyen tinh: tim khoang [BPi, BPi+1] chua c roi tinh dung Cong thuc 2
    # Lam tron thanh so nguyen theo quy dinh cua QD 1459.
    return np.round(np.interp(c, BP_PM25, I_AQI)).astype(int)

# KIEM TRA: ham cua minh phai cho dung cac gia tri tinh tay. Neu sai, assert se bao loi ngay.
thu = {0: 0, 25: 50, 37.5: 75, 50: 100, 65: 125, 80: 150, 150: 200, 250: 300}
for pm, mong_doi in thu.items():
    kq = int(vn_aqi_pm25(pm))
    assert kq == mong_doi, f"PM2.5={pm}: ham tra {kq}, phai la {mong_doi}"
print("Ham vn_aqi_pm25 dung voi tat ca", len(thu), "diem kiem tra tinh tay.")

# 1) VN_AQI thanh phan PM2.5 (so nguyen 0-500)
df["vn_aqi_pm25"] = vn_aqi_pm25(df["pm25"])

# 2) Gan nhan muc theo khoang AQI cua QD 1459: 0-50, 51-100, 101-150, 151-200, 201-300, 301-500
df["vn_aqi_cat"] = pd.cut(
    df["vn_aqi_pm25"],
    bins=[-1, 50, 100, 150, 200, 300, 500],
    labels=["Tot", "Trung binh", "Kem", "Xau", "Rat xau", "Nguy hai"],
).astype(str)

# 3) Gioi han QCVN 24 gio: 50 ug/m3, rieng tu 01/01/2026 la 45 (chu thich (*) Bang 1 QCVN 05:2023)
#    np.where(dieu_kien, gia_tri_neu_dung, gia_tri_neu_sai): giong IF trong Excel, ap cho ca cot
df["qcvn_24h"] = np.where(df["date"] >= "2026-01-01", 45, 50)
df["vuot_qcvn"] = (df["pm25"] > df["qcvn_24h"]).astype(int)

# 4) Khuyen nghi 24 gio cua WHO 2021 = 15 ug/m3
df["vuot_who"] = (df["pm25"] > 15).astype(int)

# 5) Tu muc "Kem" tro len (AQI > 100): nguong canh bao chinh thuc cho nhom nhay cam
df["kem_tro_len"] = (df["vn_aqi_pm25"] > 100).astype(int)

# Ty le ngay vuot tung nguong, tren toan bo du lieu
nguong = [
    ("Vuot WHO 2021 (24h > 15)",                       df["vuot_who"]),
    ("Tu muc 'Trung binh' tro len (PM2.5 > 25)",     (df["pm25"] > 25).astype(int)),
    ("Vuot QCVN 24h (50, tu 2026: 45)",                df["vuot_qcvn"]),
    ("Tu muc 'Kem' tro len (AQI > 100)",                   df["kem_tro_len"]),
]
print(f"\n{'Nguong':52s} {'So ngay':>8s} {'Ty le':>7s}")
for ten, cot in nguong:
    print(f"{ten:52s} {int(cot.sum()):8d} {100 * cot.mean():6.1f}%")

print("\nPhan bo theo muc VN_AQI (so ngay):")
thu_tu = ["Tot", "Trung binh", "Kem", "Xau", "Rat xau", "Nguy hai"]
print(df["vn_aqi_cat"].value_counts().reindex(thu_tu).fillna(0).astype(int))

BUOC K: Gan thang do chinh thuc cua Viet Nam
Ham vn_aqi_pm25 dung voi tat ca 8 diem kiem tra tinh tay.

Nguong                                                So ngay   Ty le
Vuot WHO 2021 (24h > 15)                                  646   98.3%
Tu muc 'Trung binh' tro len (PM2.5 > 25)                  485   73.8%
Vuot QCVN 24h (50, tu 2026: 45)                            54    8.2%
Tu muc 'Kem' tro len (AQI > 100)                           46    7.0%

Phan bo theo muc VN_AQI (so ngay):
vn_aqi_cat
Tot           175
Trung binh    436
Kem            45
Xau             1
Rat xau         0
Nguy hai        0
Name: count, dtype: int64


**Đọc kết quả (nối về câu hỏi kinh doanh):**

1. **Hầu như mọi ngày đều vượt khuyến nghị WHO**; vấn đề không phải "có ngày nào xấu không" mà là "khi nào xấu nhiều".
2. Theo chuẩn Việt Nam, ngày cần cảnh báo (*Kém* trở lên) chỉ khoảng **7%** → biến Y `kem_tro_len` **hiếm** (mỗi tập train/test chỉ ~20–30 ngày dương tính), mô hình phân loại khó học và kết quả bất ổn: phải báo cáo trung thực. Bài toán dự báo PM2.5 liên tục không bị ảnh hưởng.
3. `vn_aqi_pm25` chỉ là chỉ số thành phần PM2.5 (cận dưới của VN_AQI thật vì trạm không đo O₃, NO₂…).
4. Ngưỡng cảnh báo "Kém" của Việt Nam bắt đầu từ **50 µg/m³**, cao hơn mốc 35,4 của thang Mỹ — nhóm dùng đúng mốc cơ quan nhà nước Việt Nam dùng để cảnh báo.

## BƯỚC J: Kiểm tra rồi ghi file 

File vẫn **chỉ có một**: `aqi_hcm_daily.csv`. So với dữ liệu sau Bước E, bảng cuối có thêm: 4 cột hướng gió, 1 cột số ngày kể từ mưa, 2 cột ngày lễ/Tết, 7 cột khí từ mô hình CAMS, 6 cột chuẩn Việt Nam.

Từ đầu đến đây mọi thứ nằm trong bộ nhớ (biến `df`); **chưa có file dữ liệu nào được ghi**. Ô dưới ghi ra **hai** file:

1. `cmt8_daily_raw.csv` — bản **thô** của trạm (PM2.5, nhiệt độ, độ ẩm và cả PM10 hỏng) trước khi ghép thời tiết và làm sạch. Giữ để người khác đối chiếu "số gốc từ trạm là gì, mình đã biến đổi gì". Đây là bằng chứng minh bạch, không dùng để phân tích.
2. `aqi_hcm_daily.csv` — file **chính, duy nhất** cả nhóm dùng cho mọi notebook sau.

**Khóa:** nếu `aqi_hcm_daily.csv` đã tồn tại và `GHI_DE_FILE_CHINH = False` thì không ghi đè, chỉ ghi ra `aqi_hcm_daily_chay_lai.csv`.

In [18]:
COT_CU = ["date", "pm25", "pm25_cov", "temperature", "temperature_cov", "humidity", "humidity_cov",
          "wind", "rain", "month", "dow", "mua", "pbl_sang_m", "ap_suat"]
COT_MOI = ["wind_dir_deg", "wind_sin", "wind_cos", "wind_sector", "ngay_tu_mua", "ky_nghi_le", "tet",
           "co_cams", "no_cams", "no2_cams", "so2_cams", "o3_cams", "o3_max_cams", "pm10_cams",
           "vn_aqi_pm25", "vn_aqi_cat", "qcvn_24h", "vuot_qcvn", "vuot_who", "kem_tro_len"]

df = df[COT_CU + COT_MOI]          # sap xep cot: cot cu truoc, cot moi sau

os.makedirs(OUT_DIR, exist_ok=True)   # tao thu muc neu chua co, khong loi neu da ton tai

# --- 1. Luu ban THO (truoc khi ghep thoi tiet va lam sach) ---
# Giu ban tho de nguoi khac doi chieu duoc: so lieu goc tu tram la gi,
# minh da bien doi nhung gi. Day la bang chung minh bach, khong phai file thua.
ban_tho = dfs["pm25"].merge(dfs["temperature"], on="date", how="outer") \
                      .merge(dfs["humidity"], on="date", how="outer")
if "pm10_df" in dir():
    ban_tho = ban_tho.merge(pm10_df, on="date", how="left")   # giu ca pm10 de chung minh sensor da chet
ban_tho = ban_tho.sort_values("date")
duong_tho = os.path.join(OUT_DIR, "cmt8_daily_raw.csv")
if os.path.exists(duong_tho) and not GHI_DE_FILE_CHINH:
    duong_tho = os.path.join(OUT_DIR, "cmt8_daily_raw_chay_lai.csv")     # KHOA: khong ghi de ban tho cu
ban_tho.to_csv(duong_tho, index=False)
print("Da luu ban THO   :", duong_tho, f"({len(ban_tho)} dong)")

# --- 2. Luu ban DA LAM SACH (file chinh dung cho moi phan tich) ---
duong_chinh = os.path.join(OUT_DIR, "aqi_hcm_daily.csv")
if os.path.exists(duong_chinh) and not GHI_DE_FILE_CHINH:
    duong_dan = os.path.join(OUT_DIR, "aqi_hcm_daily_chay_lai.csv")     # KHOA: khong ghi de file chinh
    print("File chinh da co -> KHONG ghi de, ban chay lai luu o:", duong_dan)
else:
    duong_dan = duong_chinh
    print("Da luu ban SACH  :", duong_dan)
df.to_csv(duong_dan, index=False)

print("Kich thuoc:", df.shape[0], "dong x", df.shape[1], "cot")
print("Khoang thoi gian:", df["date"].min().date(), "->", df["date"].max().date())
print("Cac cot:", list(df.columns))
